<a href="https://colab.research.google.com/github/zamanuddinkhan/Python-AI-LLM/blob/main/CrewAI_Simple_Web_Scrape.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29 langchain-openai beautifulsoup4 requests sentence-transformers faiss-cpu

In [ ]:
import requests
from bs4 import BeautifulSoup

# Correct CrewAI imports
from crewai import Agent, Task, Crew
from crewai_tools import Tool   # ✅ correct import for Tool

# RAG imports
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# LLM import
from langchain_openai import ChatOpenAI


In [ ]:
#Custom Tool
def scrape_website(url: str):
    """Scrapes text content from a webpage"""
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        r = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        text = soup.get_text(separator=" ", strip=True)
        return text[:15000]   # Avoid sending too much to the LLM
    except Exception as e:
        return f"Scraping error: {e}"

In [ ]:
scraper_tool = Tool(
    name="web_scraper",
    func=scrape_website,
    description="Scrapes text content from a webpage URL."
)

In [ ]:
embedder = SentenceTransformer("all-mpnet-base-v2")

In [ ]:
def build_faiss_index(chunks):
    embeddings = embedder.encode(chunks, show_progress_bar=False)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(np.array(embeddings))
    return index, embeddings

In [ ]:
def rag_search(query: str):
    """Returns 3 most relevant chunks based on similarity search."""
    q_emb = embedder.encode([query])
    D, I = faiss_index.search(q_emb, k=3)
    results = "\n\n".join([chunks[i] for i in I[0]])
    return results


In [ ]:
rag_tool = Tool(
    name="rag_retriever",
    func=rag_search,
    description="Retrieves the most relevant text chunks from FAISS."
)


In [ ]:
url = "https://en.wikipedia.org/wiki/Natural_language_processing"
scraped_text = scrape_website(url)


In [ ]:
print(scraped_text)

In [ ]:
chunks = [scraped_text[i:i+600] for i in range(0, len(scraped_text), 600)]


In [ ]:
faiss_index, embeddings = build_faiss_index(chunks)


In [ ]:
from google.colab import userdata
import os
os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')

In [ ]:
educator_agent = Agent(
    role="NLP Educator",
    goal="Explain NLP using retrieved context from RAG.",
    backstory="You are an NLP teaching expert who simplifies complex topics.",
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0.2),
    tools=[scraper_tool, rag_tool],
    verbose=True
)


In [ ]:
explain_task = Task(
    description="Explain NLP concepts using the scraped and retrieved RAG content.",
    expected_output="A structured NLP explanation with examples.",
    agent=educator_agent
)


In [ ]:
crew = Crew(
    agents=[educator_agent],
    tasks=[explain_task],
    verbose=True
)

final_output = crew.kickoff()
print("\n\n===== FINAL OUTPUT =====\n")
print(final_output)